# Gozcu — Model-512 egitimi

HERIDAL havadan arama-kurtarma goruntulerinde insan tespiti. Bu notebook,
512 px karolara bolunmus egitim kumesi uzerinde yolo11n'i egitir.

## Sabitlenen kararlar

- **`imgsz=512`** — egitim karosu ile cikarim karosu ayni olmali (Hafta 3, 5D.2).
  Model egitimde gordugu olcekte calistirilmazsa hedef boyutu dagilimi kayar.
- **Veri artirma ayarlarina DOKUNULMADI.** Ultralytics varsayilanlari aynen
  kaliyor (`mosaic=1.0`, `fliplr=0.5`, `flipud=0.0`). Nedeni: bu kosuda tek
  degisen sey egitimin kendisi olsun ki taban cizgisine gore olculen kazanc
  egitime atfedilebilsin. Artirma denemeleri sonraki kosuya birakildi.
- **`device=0`** — tek GPU. Notebook icinde cok GPU'lu DDP kirilgan.
- **`yolo11n.pt` baslangic agirliklari** — COCO on-egitimi uzerine transfer
  ogrenme; sifirdan egitim bu veri buyuklugunde tercih edilmiyor.


## UYARI — bu notebook'un sayilari taban cizgisiyle KARSILASTIRILAMAZ

Bu notebook'un urettigi mAP ve recall sayilari **bizim taban cizgimizle
karsilastirilamaz.** Kaggle'daki dogrulama 512x512 karolar uzerinde yapilir;
bizim taban cizgimiz ise 4000x3000 tam goruntuler uzerinde SAHI ile olculdu.
Ikisi farkli olcum birimleridir.

Kapi kararini veren tek olcum, yerelde `scripts/10_test_taban_cizgisi.py` ile
yapilan olcumdur. Buradaki val mAP yalnizca egitimin yakinsayip yakinsamadigini
ve asiri ogrenme olup olmadigini gormek icindir.


In [ ]:
# Kaggle'da Internet = On olmali; kapaliysa bu hucre paketi indiremez.
!pip install ultralytics -q

import ultralytics
print('ultralytics', ultralytics.__version__)


## Veri yapisi — /kaggle/input salt okunur

Ultralytics etiket onbellegini `Path(etiket_dosyasi).parent.with_suffix('.cache')`
yoluna yazar; yani `labels/train/x.txt` icin onbellek `labels/train.cache` olur ve
**`labels/` dizininin icine** dusen bir dosyadir. `/kaggle/input` salt okunur
oldugundan, `labels/` dogrudan oraya baglanirsa egitim bu yazma denemesinde patlar.

**Secilen cozum: split seviyesinde symlink.** `images/` ve `labels/` dizinlerinin
kendisi `/kaggle/working/veri` altinda gercek (yazilabilir) dizinlerdir; yalnizca
iclerindeki `train` ve `val` `/kaggle/input`'a symlink'tir. Boylece onbellek
yazilabilir gercek dizine duser ve 1,4 GB veri kopyalanmaz.

Kopyalama alternatifi de calisirdi, ama `/kaggle/working` diskini 1,4 GB sisirir
ve her kosuda dakikalar harcardi.


In [ ]:
import shutil
from pathlib import Path

GIRDI = Path('/kaggle/input/gozcu-karo-512')
CALISMA = Path('/kaggle/working/veri')

if CALISMA.exists():
    shutil.rmtree(CALISMA)

for tur in ('images', 'labels'):
    (CALISMA / tur).mkdir(parents=True)          # GERCEK dizin (yazilabilir)
    for bolum in ('train', 'val'):
        (CALISMA / tur / bolum).symlink_to(GIRDI / tur / bolum)   # symlink

# data.yaml yeniden uretiliyor: yereldeki kopya mutlak YEREL yollar iceriyor.
# path olarak /kaggle/input degil /kaggle/working/veri veriliyor -- onbellegin
# yazilabilir dizine dusmesi buna bagli.
# Sinif adi 'human': yereldeki data.yaml'i scripts/11_karo_veri_hazirla.py
# uretiyor ve oraya 'human' yaziyor. Elle yazilmis deger degil, URETECIN
# degeri esas alinir; yoksa script bir daha kosuldugunda catisma geri gelir.
# Kaggle'a yuklenen data.yaml okunmuyor (bu hucre yenisini uretiyor),
# o yuzden kumeyi yeniden yuklemek gerekmiyor.
yaml_yolu = CALISMA / 'data.yaml'
yaml_yolu.write_text(
    f'path: {CALISMA}\n'
    'train: images/train\n'
    'val: images/val\n'
    'nc: 1\n'
    "names: ['human']\n"
)
print(yaml_yolu.read_text())


In [ ]:
# Egitimi baslatmadan once yapiyi ucuza dogrula: 100 dakikalik bir kosunun
# ilk saniyesinde patlamasindansa burada patlasin.
import glob, os
from ultralytics.data.utils import img2label_paths

for bolum in ('train', 'val'):
    goruntuler = sorted(glob.glob(str(CALISMA / 'images' / bolum / '*.jpg')))
    etiketler = img2label_paths(goruntuler)
    eksik = [e for e in etiketler if not os.path.isfile(e)]
    onbellek = Path(etiketler[0]).parent.with_suffix('.cache')
    yazilabilir = os.access(onbellek.parent, os.W_OK)
    print(f'{bolum}: {len(goruntuler)} goruntu | eksik etiket {len(eksik)} | '
          f'onbellek {onbellek} | yazilabilir {yazilabilir}')
    assert goruntuler, f'{bolum} bolumunde goruntu bulunamadi'
    assert not eksik, f'{bolum}: {len(eksik)} goruntunun etiketi yok'
    assert yazilabilir, 'Onbellek dizini yazilabilir degil -- symlink kurulumu hatali'

print('Beklenen: train 7924 goruntu, val 3160 goruntu.')
print('Yapi dogru.')


## Egitim

`patience=20`: 20 epoch boyunca val metrigi iyilesmezse erken durur, yani 100
epoch bir ust sinirdir. `seed=0` ve `deterministic=True` kosuyu tekrarlanabilir kilar.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
sonuc = model.train(
    data='/kaggle/working/veri/data.yaml',
    epochs=100,
    imgsz=512,
    batch=16,
    patience=20,
    seed=0,
    deterministic=True,
    device=0,
    project='/kaggle/working/egitim',
    name='model512',
    exist_ok=True,
)


In [ ]:
from pathlib import Path

best = Path('/kaggle/working/egitim/model512/weights/best.pt')
son = Path('/kaggle/working/egitim/model512/weights/last.pt')

for yol in (best, son):
    if yol.is_file():
        print(f'{yol}  ->  {yol.stat().st_size / 1024 / 1024:.2f} MB')
    else:
        print(f'{yol}  ->  YOK')

print()
print('Indirilecek dosya: best.pt')
print('Yerelde konacagi yer: agirliklar/model512_best.pt (bkz. egitim/README.md)')
